# Within-Dataset Benchmark: Ridge, Random Forest, LightGBM, GraphDRP, SimpleLinearNN

This notebook runs the first benchmark stage using the official IMPROVE within-dataset splits. It starts with five models:

- `ridge`
- `random_forest`
- `lightgbm`
- `graphdrp`
- `simple_linear_nn`

The code is kept in `within_dataset_4models.py` so the model registry can be extended later without turning the notebook into a maze.

In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd()
if ROOT.name == "new_notebook":
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "new_notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import within_dataset_4models
importlib.reload(within_dataset_4models)
from within_dataset_4models import (
    CELL_ID_COL,
    get_lincs_symbol_list,
    make_default_config,
    read_gene_expression,
    read_response,
    response_for_split,
    run_within_dataset_benchmark,
    select_gene_columns,
    summarize_results,
)

cfg = make_default_config(ROOT)
cfg


## Config

Default is a quick smoke test on `CCLE`, fold `0`. For the full paper-style within-dataset run, switch to all five datasets and ten folds.

In [ ]:
# Smoke test / current experiment
cfg.datasets = ["CCLE"]
cfg.folds = list(range(3))

# Full paper-style within-dataset benchmark. Uncomment when ready.
# cfg.datasets = ["gCSI", "CCLE", "GDSCv2", "GDSCv1", "CTRPv2"]
# cfg.folds = list(range(10))

cfg.models = ["ridge", "random_forest", "lightgbm", "graphdrp", "simple_linear_nn"]


# Required for GraphDRP/SimpleLinearNN/RF-style preprocessing.
# Uses the built-in LINCS/L1000 landmark list instead of improvelib.
cfg.use_lincs_symbol_genes = True

# Mordred can still be capped for tabular models; top_ge_features is ignored while use_lincs_symbol_genes=True.
cfg.top_ge_features = 512
cfg.top_mordred_features = 512

cfg.graphdrp_epochs = 150
cfg.graphdrp_patience = 20
cfg.simple_nn_epochs = 300
cfg.simple_nn_patience = 50
cfg.simple_nn_model = "default"

# Use these for a very quick debug run, then set back to None.
cfg.max_train_rows = None
cfg.max_eval_rows = None

cfg


## Validate GraphDRP Preprocessing

GraphDRP-style preprocessing uses the built-in LINCS/L1000 landmark list. This check must show `958` mapped gene-expression columns on the current CSA data; `512` means the old top-variance fallback is still being used.


In [ ]:
lincs_symbols = get_lincs_symbol_list(required=True)
response_check = read_response(cfg)
gene_expression_check = read_gene_expression(cfg)
train_check = response_for_split(cfg, response_check, cfg.datasets[0], cfg.folds[0], "train")
ge_cols_check = select_gene_columns(cfg, gene_expression_check, train_check[CELL_ID_COL].unique())

print(f"built-in LINCS_SYMBOL: {len(lincs_symbols)}")
print(f"Mapped gene-expression columns: {len(ge_cols_check)}")
assert len(lincs_symbols) == 976
assert len(ge_cols_check) == 958, (
    f"Expected 958 mapped LINCS gene-expression columns, got {len(ge_cols_check)}. "
    "If this says 512, reload within_dataset_4models.py and make sure the helper module is current."
)


## Run Benchmark

GraphDRP requires `torch`, `torch-geometric`, and `rdkit`. SimpleLinearNN requires `torch`. If dependencies are missing, the runner records the model as skipped instead of stopping the whole benchmark.

In [ ]:
results = run_within_dataset_benchmark(cfg)
display(results)

## Summary

In [ ]:
summary = summarize_results(results, cfg.out_dir)
display(summary)